# LightGBM Classification — proper train/test + k-fold CV tuning

**Fixes the "tuned on test" problem.** Protocol:
1. Hold out a **test set once** (stratified by dataset), never touched during tuning.
2. Inside the remaining **train** portion, run **5-fold cross-validation** to select
   hyperparameters — every hyperparameter config is scored on validation folds, not test.
3. Evaluate the chosen config on the held-out test set **exactly once** and report that.

This also gives **error bars** (std across folds), which lets you say whether models
are statistically distinguishable rather than just numerically close.

Metric for selection: mean CV **large-gap decision accuracy** (top-25% true-gap
accuracy) — the number that matters for the tie-breaker — not overall accuracy.

## Setup

In [1]:
!pip install -q lightgbm scikit-learn scipy requests pandas numpy joblib
import os; os.makedirs('artifacts',exist_ok=True)
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [2]:
# === shared reconstruction + DIFFERENCE-target logic ===
import io, re, requests
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr

REPO="https://raw.githubusercontent.com/yoshikodes/KVCacheCompression/main/Data"
SETTINGS=["h2o_20","h2o_40","h2o_60","kvquant_2bit","kvquant_3bit","kvquant_4bit"]
H2O=["h2o_20","h2o_40","h2o_60"]; KVQ=["kvquant_2bit","kvquant_3bit","kvquant_4bit"]
DATASETS=["gsm8k","arc_challenge","hellaswag","wikitext103"]
FNAME={"h2o_20":"h2o_budget_20pct","h2o_40":"h2o_budget_40pct","h2o_60":"h2o_budget_60pct",
       "kvquant_2bit":"kvquant_2bit","kvquant_3bit":"kvquant_3bit","kvquant_4bit":"kvquant_4bit"}
IDX=["question_index","item_index","chunk_index"]
BASE_THRESH=500.0; RS=42; TEST=0.30

# The 9 H2O x KVQuant pairs -- these ARE the decision the tie-breaker makes.
PAIRS=[(h,k) for h in H2O for k in KVQ]
PAIR_NAMES=[f"{h}__vs__{k}" for h,k in PAIRS]

def _get(u):
    r=requests.get(u,timeout=30); r.raise_for_status(); return pd.read_csv(io.StringIO(r.text))

def reconstruct(verbose=True):
    frames=[]
    for s in SETTINGS:
        for ds in DATASETS:
            try: df=_get(f"{REPO}/{FNAME[s]}_{ds}_per_prompt.csv")
            except Exception as e:
                if verbose: print("skip",s,ds,e); continue
            frames.append(pd.DataFrame({"dataset":ds,"setting":s,
                "prompt":df["prompt"].astype(str),
                "perplexity":pd.to_numeric(df["perplexity"],errors="coerce")}))
    long=pd.concat(frames,ignore_index=True).dropna(subset=["perplexity"])
    wide=long.pivot_table(index=["dataset","prompt"],columns="setting",
                          values="perplexity",aggfunc="mean").reset_index()
    wide.columns.name=None
    wide=wide.dropna(subset=SETTINGS)
    brows=[]
    for ds in DATASETS:
        try: b=_get(f"{REPO}/kvquant_baseline_full_precision_{ds}_per_prompt.csv")
        except Exception: continue
        brows.append(pd.DataFrame({"prompt":b["prompt"].astype(str),
            "baseline_ppl":pd.to_numeric(b["perplexity"],errors="coerce")}))
    base=pd.concat(brows,ignore_index=True).dropna().drop_duplicates("prompt")
    wide=wide.merge(base,on="prompt",how="left")
    n0=len(wide)
    wide=wide[wide["baseline_ppl"].notna() & (wide["baseline_ppl"]<=BASE_THRESH)].reset_index(drop=True)
    if verbose:
        print(f"reconstructed {n0}, {len(wide)} after baseline<={BASE_THRESH}")
        print(wide["dataset"].value_counts().to_string())
    return wide

def split(df):
    tr,te=[],[]
    for ds,g in df.groupby("dataset"):
        a,b=train_test_split(g,test_size=TEST,random_state=RS,shuffle=True)
        tr.append(a); te.append(b)
    return (pd.concat(tr).sample(frac=1,random_state=RS).reset_index(drop=True),
            pd.concat(te).sample(frac=1,random_state=RS).reset_index(drop=True))

def diff_targets(df):
    """Target = log(H2O_ppl) - log(KVQ_ppl) for each of the 9 pairs.
    Negative => H2O is better (lower perplexity); positive => KVQuant better."""
    out=np.zeros((len(df),len(PAIRS)))
    for j,(h,k) in enumerate(PAIRS):
        out[:,j]=np.log(df[h].values)-np.log(df[k].values)
    return out

def evaluate_diff(y_true,y_pred):
    rows=[]
    for j,name in enumerate(PAIR_NAMES):
        yt,yp=y_true[:,j],y_pred[:,j]
        # decision accuracy: did we get the SIGN right? (which method wins)
        acc=np.mean(np.sign(yt)==np.sign(yp))
        rows.append({"pair":name,"MAE":mean_absolute_error(yt,yp),
            "R2":r2_score(yt,yp),"sign_acc":acc,
            "Spearman":spearmanr(yt,yp).correlation})
    per=pd.DataFrame(rows)
    per.loc[len(per)]={"pair":"OVERALL","MAE":per.MAE.mean(),"R2":per.R2.mean(),
        "sign_acc":per.sign_acc.mean(),"Spearman":per.Spearman.mean()}
    return per

def large_gap_analysis(y_true,y_pred,quantiles=(0.5,0.75,0.9)):
    """KEY analysis: accuracy on prompts where the gap is LARGE (where getting it
    right actually matters). Near-ties are cheap to miss; big gaps are not."""
    absg=np.abs(y_true.ravel())
    sign_correct=(np.sign(y_true)==np.sign(y_pred)).ravel()
    print("Decision accuracy by size of the true H2O-vs-KVQ gap:")
    print(f"  {'gap percentile':<22}{'threshold':>10}{'n':>8}{'accuracy':>10}")
    print(f"  {'all pairs':<22}{'-':>10}{len(absg):>8}{sign_correct.mean():>10.4f}")
    for q in quantiles:
        thr=np.quantile(absg,q)
        m=absg>=thr
        print(f"  {'top '+str(int((1-q)*100))+'% largest gaps':<22}{thr:>10.3f}{m.sum():>8}{sign_correct[m].mean():>10.4f}")
    # regret in log-ppl when we pick wrong
    wrong=~sign_correct
    regret=absg[wrong]
    print(f"\n  mean regret on wrong picks (log-ppl): {regret.mean():.4f}")
    print(f"  median regret overall (log-ppl)     : {np.median(np.where(sign_correct,0,absg)):.4f}")


## Reconstruct + hold out test set ONCE

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

# Reconstruct the full dataset from the repo (24 compressed + 4 baseline CSVs),
# apply the baseline<=500 filter. `wide` = one row per prompt with all 6 settings.
wide = reconstruct()

# ============================================================================
# STEP 1: Hold out a TEST set ONCE. This is the core of honest evaluation.
# The test set will NOT be looked at during hyperparameter tuning — only at the
# very end, a single time. That prevents "tuning on the test set", which makes
# reported numbers look better than they really are.
# ============================================================================
TEST_FRAC = 0.15   # keep 15% of prompts aside for the final test. 85% is used for
                   # training + cross-validation (preserves training data, which
                   # matters on a small dataset).

# We split SEPARATELY within each dataset (stratified), so the test set has the
# same mix of gsm8k/arc/hellaswag/wikitext as the whole. Without this, a random
# split might put, say, most wikitext prompts on one side.
tr_parts, te_parts = [], []
for ds, g in wide.groupby("dataset"):
    a, b = train_test_split(g, test_size=TEST_FRAC, random_state=RS, shuffle=True)
    tr_parts.append(a)   # this dataset's train+val portion
    te_parts.append(b)   # this dataset's test portion

# trainval_df = the 85% we tune on.  test_df = the 15% we touch only once at the end.
trainval_df = pd.concat(tr_parts).sample(frac=1, random_state=RS).reset_index(drop=True)
test_df     = pd.concat(te_parts).sample(frac=1, random_state=RS).reset_index(drop=True)

print("train+val:", len(trainval_df), "| test (held out, untouched until the end):", len(test_df))
print("per-dataset train+val:", trainval_df["dataset"].value_counts().to_dict())

reconstructed 3150, 3109 after baseline<=500.0
dataset
hellaswag        1023
arc_challenge     982
gsm8k             962
wikitext103       142
train+val: 2485 | test (held out, untouched until the end): 624
per-dataset train+val: {'hellaswag': 818, 'arc_challenge': 785, 'gsm8k': 769, 'wikitext103': 113}


## Features + labels

In [4]:
import re

# --------------------------------------------------------------------------
# FEATURES: cheap numeric descriptions of each prompt's text. The model uses
# these to predict which compression method wins. (No fine-tuning here — just
# hand-built features fed to gradient-boosted trees.)
# --------------------------------------------------------------------------
def feats(p):
    s = str(p)
    t = s.split()                      # words (whitespace tokens)
    nt = max(len(t), 1)                # #tokens (avoid divide-by-zero)
    nc = max(len(s), 1)                # #characters
    return {
        "n_char": len(s),                                  # prompt length in chars
        "n_tok": len(t),                                   # prompt length in words
        "ttr": len(set(t)) / nt,                           # type-token ratio (vocab diversity / repetition)
        "digit_ratio": sum(c.isdigit() for c in s) / nc,   # how numeric the text is
        "punct_ratio": sum(not c.isalnum() and not c.isspace() for c in s) / nc,  # punctuation density
        "upper_ratio": sum(c.isupper() for c in s) / nc,   # uppercase density
        "avg_tok": nc / nt,                                # average word length
        "num_count": len(re.findall(r"\d+", s)),          # count of numbers in the prompt
        "has_q": int("?" in s),                            # is it a question?
    }

def fmat(df):
    # Turn a dataframe of prompts into a feature matrix (one row per prompt).
    X = pd.DataFrame([feats(p) for p in df["prompt"]])
    # Add the prompt's UNCOMPRESSED (baseline) perplexity as a feature. It's a very
    # strong anchor: a prompt that's high-perplexity uncompressed tends to stay high
    # compressed. Available at inference time in your system, so it's fair to use.
    X["baseline_ppl"] = np.log(df["baseline_ppl"].values)  # log = tames the huge range
    return X

# --------------------------------------------------------------------------
# LABELS: for each of the 9 H2O-vs-KVQuant pairs, which method wins on this prompt?
# diff_targets returns log(H2O_ppl) - log(KVQ_ppl) per pair. Negative => H2O has
# LOWER perplexity => H2O wins => label 1. Positive => KVQuant wins => label 0.
# --------------------------------------------------------------------------
def labels(df):
    return (diff_targets(df) < 0).astype(int)   # shape (n_prompts, 9), values 0/1

# --------------------------------------------------------------------------
# The metric that matters for the tie-breaker: accuracy on the LARGE-GAP prompts,
# where H2O and KVQuant genuinely differ (so picking wrong actually costs quality).
# Near-ties are excluded because getting those wrong barely matters.
# --------------------------------------------------------------------------
def large_gap_acc(df, proba, q=0.75):
    td = diff_targets(df)                 # true signed differences, shape (n, 9)
    gap = np.abs(td).ravel()              # magnitude of each difference (flattened)
    # was the predicted winner correct? (proba>=0.5 predicts H2O; true H2O win = td<0)
    correct = ((proba >= 0.5).astype(int) == (td < 0).astype(int)).ravel()
    thr = np.quantile(gap, q)             # e.g. q=0.75 -> the top 25% biggest gaps
    m = gap >= thr                        # mask selecting only the large-gap pairs
    return correct[m].mean(), correct.mean()   # (large-gap accuracy, overall accuracy)

## Hyperparameter search via 5-fold CV

Each config is scored by mean large-gap accuracy across 5 folds of the train+val
data. The test set is NOT used here. Add/adjust configs in CANDIDATES.

In [5]:
import lightgbm as lgb
import itertools
from sklearn.model_selection import RepeatedStratifiedKFold

# ============================================================================
# STEP 2: choose hyperparameters using REPEATED 5-fold cross-validation on the
# train+val data only. The test set is NOT touched here.
#
# Plain k-fold splits train+val into 5 folds, validates on each once (5 scores).
# REPEATED k-fold does that whole procedure N_REPEATS times, each with a DIFFERENT
# random partition into 5 folds. So:
#   - within each repeat: every prompt is validated exactly once (clean coverage)
#   - across repeats: the partition changes, averaging out the "luck" of any one
#     particular way of cutting the data into folds.
# Total validation scores per config = N_FOLDS * N_REPEATS. Averaging over all of
# them gives a more stable estimate and a more honest error bar.
# ============================================================================

# ---- FIRST-PRINCIPLES hyperparameter grid --------------------------------
# Ranges chosen by convention for a SMALL tabular dataset (centered on LightGBM
# defaults, extended toward stronger regularization because data is limited).
# Defined WITHOUT reference to any prior results, so CV selection is unbiased.
NUM_LEAVES_GRID   = [4, 8, 16, 31, 63]    # tree complexity: simple -> above-default
MIN_CHILD_GRID    = [20, 50, 100, 200]    # leaf-size floor: default -> heavy reg
N_ESTIMATORS_GRID = [100, 200, 400]       # boosting rounds (paired with lr=0.03)

CANDIDATES = [
    {"num_leaves": nl, "min_child_samples": mc, "n_estimators": ne}
    for nl, mc, ne in itertools.product(NUM_LEAVES_GRID, MIN_CHILD_GRID, N_ESTIMATORS_GRID)
]

# Settings held FIXED across all candidates (not tuned here).
FIXED = dict(learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
             random_state=RS, verbose=-1)

N_FOLDS   = 5   # folds per repeat. 5 is standard.
N_REPEATS = 3   # how many times to repeat the whole k-fold with a fresh partition.
                # Higher = more stable estimate + tighter error bars, but N_REPEATS x
                # more compute. 3 is a good balance; set to 1 for plain k-fold.

print(f"{len(CANDIDATES)} candidate configs, "
      f"{N_FOLDS}-fold CV x {N_REPEATS} repeats = "
      f"{N_FOLDS*N_REPEATS} val scores per config\n")

def cv_score(cfg):
    """Return all (N_FOLDS*N_REPEATS) per-fold validation scores for one config."""
    # RepeatedStratifiedKFold: runs StratifiedKFold N_REPEATS times, each with a
    # different shuffle. Every fold still keeps the dataset mix (stratified).
    rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS,
                                   random_state=RS)
    strat = trainval_df["dataset"].values   # stratify folds by dataset name

    lg_scores, ov_scores = [], []
    # rskf.split yields N_FOLDS*N_REPEATS (train_idx, val_idx) pairs in total.
    for tr_idx, va_idx in rskf.split(trainval_df, strat):
        tr = trainval_df.iloc[tr_idx]   # 4 folds -> training
        va = trainval_df.iloc[va_idx]   # 1 fold  -> validation
        Xtr, Xva = fmat(tr), fmat(va)
        ytr = labels(tr)

        # Train one classifier PER pair (9 total); collect P(H2O wins) on the val fold.
        proba = np.column_stack([
            lgb.LGBMClassifier(**cfg, **FIXED)
               .fit(Xtr.values, ytr[:, j])
               .predict_proba(Xva.values)[:, 1]
            for j in range(len(PAIR_NAMES))
        ])
        lg, ov = large_gap_acc(va, proba)   # score this fold
        lg_scores.append(lg); ov_scores.append(ov)
    return np.array(lg_scores), np.array(ov_scores)

# Score every candidate. Selection metric = mean large-gap accuracy across ALL
# folds and repeats; std is the error bar (now reflects partition variation too).
results = []
for i, cfg in enumerate(CANDIDATES, 1):
    lg, ov = cv_score(cfg)
    results.append((cfg, lg.mean(), lg.std(), ov.mean()))
    print(f"[{i:>2}/{len(CANDIDATES)}] {str(cfg):<52} "
          f"CV large-gap={lg.mean():.4f}±{lg.std():.4f}  overall={ov.mean():.4f}")

# Pick the config with the highest MEAN repeated-CV large-gap accuracy.
# (Chosen using only train+val data — the test set had no say.)
best_cfg = max(results, key=lambda r: r[1])[0]

# Show the top few for context.
results_sorted = sorted(results, key=lambda r: r[1], reverse=True)
print("\nTop 5 configs by repeated-CV large-gap accuracy:")
for cfg, lgm, lgs, ovm in results_sorted[:5]:
    print(f"  {str(cfg):<52} {lgm:.4f}±{lgs:.4f}")
print("\nBEST config (chosen by repeated CV, test set never used):", best_cfg)

60 candidate configs to search via 5-fold CV

[ 1/60] {'num_leaves': 4, 'min_child_samples': 20, 'n_estimators': 100} CV large-gap=0.8291±0.0143  overall=0.6901
[ 2/60] {'num_leaves': 4, 'min_child_samples': 20, 'n_estimators': 200} CV large-gap=0.8225±0.0136  overall=0.6897
[ 3/60] {'num_leaves': 4, 'min_child_samples': 20, 'n_estimators': 400} CV large-gap=0.8166±0.0124  overall=0.6865
[ 4/60] {'num_leaves': 4, 'min_child_samples': 50, 'n_estimators': 100} CV large-gap=0.8288±0.0155  overall=0.6898
[ 5/60] {'num_leaves': 4, 'min_child_samples': 50, 'n_estimators': 200} CV large-gap=0.8223±0.0155  overall=0.6877
[ 6/60] {'num_leaves': 4, 'min_child_samples': 50, 'n_estimators': 400} CV large-gap=0.8181±0.0162  overall=0.6882
[ 7/60] {'num_leaves': 4, 'min_child_samples': 100, 'n_estimators': 100} CV large-gap=0.8290±0.0150  overall=0.6886
[ 8/60] {'num_leaves': 4, 'min_child_samples': 100, 'n_estimators': 200} CV large-gap=0.8207±0.0106  overall=0.6866
[ 9/60] {'num_leaves': 4, 'min_c

## Final evaluation — test set, touched ONCE

Train the chosen config on ALL train+val data, evaluate on the held-out test set.
This is the number to report.

In [6]:
# ============================================================================
# FULL EVALUATION + RESULTS (run after the CV search cell that sets `best_cfg`)
# Trains the CV-chosen config on all train+val data, evaluates the held-out TEST
# set once, and prints every result: per-pair table, overall, gap, and the
# true-gap / confidence breakdowns.
# ============================================================================
import joblib
from sklearn.metrics import accuracy_score, roc_auc_score

# ---- Train final model: 9 classifiers (one per H2O-vs-KVQ pair) on ALL train+val ----
Xtv, Xte = fmat(trainval_df), fmat(test_df)
ytv, yte = labels(trainval_df), labels(test_df)
clfs = [lgb.LGBMClassifier(**best_cfg, **FIXED).fit(Xtv.values, ytv[:, j])
        for j in range(len(PAIR_NAMES))]

# Predicted P(H2O wins) for train+val and for the held-out test set.
proba_tv = np.column_stack([c.predict_proba(Xtv.values)[:, 1] for c in clfs])
proba_te = np.column_stack([c.predict_proba(Xte.values)[:, 1] for c in clfs])

# Save the model (on Colab, download or copy to Drive — artifacts/ is wiped on disconnect).
joblib.dump({"clfs": clfs, "config": best_cfg}, "artifacts/clf_01_kfold_final.joblib")

print("Config used (chosen by CV, test never involved):", best_cfg)

# ---- 1) PER-PAIR TABLE: accuracy + AUC for each of the 9 pairs, train+val vs test ----
def per_pair(y, pred, proba):
    rows = []
    for j, name in enumerate(PAIR_NAMES):
        try:    auc = roc_auc_score(y[:, j], proba[:, j])
        except ValueError: auc = float("nan")   # a pair with only one class present
        rows.append({"pair": name,
                     "acc": accuracy_score(y[:, j], pred[:, j]),
                     "auc": auc})
    return pd.DataFrame(rows)

pred_tv = (proba_tv >= 0.5).astype(int)
pred_te = (proba_te >= 0.5).astype(int)
tab_tv = per_pair(ytv, pred_tv, proba_tv).rename(columns={"acc": "acc_trainval", "auc": "auc_trainval"})
tab_te = per_pair(yte, pred_te, proba_te).rename(columns={"acc": "acc_test", "auc": "auc_test"})
table = tab_tv.merge(tab_te, on="pair")
# append an OVERALL row (mean across pairs)
table.loc[len(table)] = {"pair": "OVERALL",
    "acc_trainval": table.acc_trainval.mean(), "auc_trainval": table.auc_trainval.mean(),
    "acc_test": table.acc_test.mean(),         "auc_test": table.auc_test.mean()}

print("\n=== PER-PAIR RESULTS (train+val vs held-out test) ===")
print(table.round(4).to_string(index=False))

# ---- 2) OVERALL + LARGE-GAP + GENERALIZATION GAP ----
def overall_and_gap(df, proba, label):
    lg, ov = large_gap_acc(df, proba)   # (top-25% true-gap accuracy, overall accuracy)
    print(f"[{label}] overall acc={ov:.4f}  |  top-25% large-gap acc={lg:.4f}")
    return ov

print("\n=== FINAL RESULT (test touched exactly once) ===")
ov_tv = overall_and_gap(trainval_df, proba_tv, "TRAIN+VAL")
ov_te = overall_and_gap(test_df,     proba_te, "TEST")
print(f"train-minus-test overall gap: {ov_tv - ov_te:.4f}  (small = good generalization)")

# ---- 3) TWO BREAKDOWNS on the held-out test set ----
def full_breakdown(df, proba):
    td   = diff_targets(df)                                  # true signed differences per pair
    gap  = np.abs(td).ravel()                                # magnitude = how much methods differ
    conf = np.abs(proba - 0.5).ravel()                       # model confidence = |P - 0.5|
    correct = ((proba >= 0.5).astype(int) == (td < 0).astype(int)).ravel()

    # VIEW 1 — by TRUE gap size. Uses ground truth => EVALUATION metric (not usable live).
    # Answers: is the model right where the decision actually matters?
    print("\nBy TRUE gap size (how much the two methods actually differ):")
    for q in (0.5, 0.75, 0.9):
        thr = np.quantile(gap, q); m = gap >= thr
        print(f"  top {int((1-q)*100)}% largest gaps   n={m.sum():<6} acc={correct[m].mean():.4f}")

    # VIEW 2 — by model CONFIDENCE. No ground truth needed => usable at DEPLOYMENT.
    # This is your gating signal: trust the pick when the model is confident.
    print("By model CONFIDENCE (how sure the model is — usable at deployment):")
    for q in (0.5, 0.75, 0.9):
        thr = np.quantile(conf, q); m = conf >= thr
        print(f"  top {int((1-q)*100)}% most confident n={m.sum():<6} acc={correct[m].mean():.4f}")

print("\n=== HELD-OUT TEST BREAKDOWN ===")
full_breakdown(test_df, proba_te)

Config used (chosen by CV, test never involved): {'num_leaves': 4, 'min_child_samples': 20, 'n_estimators': 100}

=== PER-PAIR RESULTS (train+val vs held-out test) ===
                    pair  acc_trainval  auc_trainval  acc_test  auc_test
h2o_20__vs__kvquant_2bit        0.7759        0.8474    0.7853    0.8460
h2o_20__vs__kvquant_3bit        0.8109        0.8421    0.8045    0.8175
h2o_20__vs__kvquant_4bit        0.8306        0.8245    0.8333    0.8646
h2o_40__vs__kvquant_2bit        0.6805        0.7301    0.6763    0.6986
h2o_40__vs__kvquant_3bit        0.6282        0.6968    0.6058    0.6167
h2o_40__vs__kvquant_4bit        0.6728        0.6869    0.6442    0.6556
h2o_60__vs__kvquant_2bit        0.7537        0.7253    0.7452    0.6900
h2o_60__vs__kvquant_3bit        0.5944        0.6411    0.6010    0.5761
h2o_60__vs__kvquant_4bit        0.6004        0.6630    0.5048    0.5289
                 OVERALL        0.7053        0.7397    0.6889    0.6993

=== FINAL RESULT (test touch

## How to read this

- The **CV table** is where hyperparameters are chosen — by mean validation large-gap
  accuracy, with std as an error bar. If two configs' means are within ~1 std, they're
  statistically indistinguishable; prefer the simpler (more regularized) one.
- The **TEST** number is reported once, after selection. Because the test set never
  influenced tuning, it's an honest estimate of real-world performance.
- Compare this honest test number to your earlier tuned-on-test number: they should
  be close (your tuning was coarse), which *confirms* the earlier results weren't
  meaningfully inflated — but now the protocol is defensible.